# LLM Prompt Runner

Run prompts on different LLM models (Colab AI, OpenAI). Prompts can come from text or TXT files (supports multi-line prompts separated by `---`).

In [ ]:
# Install dependencies
%pip install openai -q

from datetime import datetime
# Import integrations and utilities
from integrations.colab_llm_provider import ColabLLMProvider
from integrations.openai_llm_provider import OpenAILLMProvider
from execution.input_prompt_reader import read_from_file
from execution.output_result_generator import save_results
from constants.constants import (
    MODEL_PROVIDERS,
    MODEL_KEYS,
    PROMPTS_FILE,
    TEMPERATURE,
    TOP_P,
    MAX_TOKENS,
)

## Execution

### 0. Available models

In [ ]:
print("Available models (use MODEL_KEYS[i] in step 1):")
for i, k in enumerate(MODEL_KEYS):
    print(f"  [{i}] {k} ({MODEL_PROVIDERS[k]})")

### 1. Choose model and decoding setup

In [ ]:
MODEL = MODEL_KEYS[0]

print(f"Selected model: {MODEL}")
print(f"Decoding setup: temperature={TEMPERATURE}, top_p={TOP_P}, max_tokens={MAX_TOKENS}")

### 2. Define prompt(s)

Read from TXT file in `prompts/` (prompts separated by `---`; each prompt can be multi-line).

In [ ]:
PROMPTS = read_from_file(PROMPTS_FILE)

### 3. Run and save

In [ ]:
def _get_llm(model_key: str):
    """Resolve model_key to provider and return the LLM instance."""
    if model_key not in MODEL_PROVIDERS:
        raise ValueError(f"Model '{model_key}' not found. Available: {list(MODEL_PROVIDERS.keys())}")
    provider = MODEL_PROVIDERS[model_key]
    if provider == "colab":
        return ColabLLMProvider(model_key)
    if provider == "openai":
        return OpenAILLMProvider(model_key)
    raise ValueError(f"Unsupported provider: {provider}")

def _format_result(prompt, response, model_key, model_name, start_time, duration_seconds, success, error=None):
    """Build the result dict for execute_prompt (success or error)."""
    return {
        "prompt": prompt,
        "response": response,
        "model_key": model_key,
        "model_name": model_name,
        "timestamp": start_time.isoformat(),
        "duration_seconds": duration_seconds,
        "success": success,
        "error": error,
    }


def _execute_prompt(model_key: str, prompt: str):
    """Execute a single prompt using the specified model (config via env vars)."""
    llm = _get_llm(model_key)
    
    start_time = datetime.now()
    
    try:
        response = llm.generate(
            prompt,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
        )
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        return _format_result(prompt, response, model_key, llm.model_name, start_time, duration, success=True)
        
    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        return _format_result(prompt, None, model_key, llm.model_name, start_time, duration, success=False, error=str(e))


results = []
for prompt in PROMPTS:
    result = _execute_prompt(MODEL, prompt)
    results.append(result)

if results:
    output_path = save_results(results, MODEL, results[0]['model_name'])

## Configuration

### Adding new models

Add model details to the appropriate integration file:
- For Colab: Edit `integrations/colab_llm_provider.py` → Add to `COLAB_MODELS`
- For OpenAI: Edit `integrations/openai_llm_provider.py` → Add to `OPENAI_MODELS`

Models are automatically discovered from integration files.

### Decoding parameters

All experiments use fixed decoding parameters: `temperature = 0.0`, `top_p = 1.0`, `max_tokens = 2048`.

### Prompt file format (TXT)

Prompts separated by `---` on a single line. Each prompt can be multi-line. (If there is no `---`, each non-empty line is treated as a separate prompt.)
```
What is the capital of Brazil?
---
Explain Python in one paragraph.
---
What is machine learning?
Give a definition and two examples.
```

### Project Structure

```
├── constants/
│   └── constants.py                # Constantes centralizadas
├── integrations/
│   ├── base_llm_provider.py        # Base class for LLM integrations
│   ├── colab_llm_provider.py       # Google Colab AI (with COLAB_MODELS)
│   └── openai_llm_provider.py      # OpenAI (with OPENAI_MODELS)
├── execution/
│   ├── input_prompt_reader.py      # Read prompts from .txt files
│   ├── output_result_generator.py  # Save results to files
│   └── runner.ipynb                # Execution notebook
├── prompts/                         # Input prompt files
└── results/                         # Output JSON results
```

### Using in Google Colab

1. Upload all directories (`constants/`, `integrations/`, `execution/`) to your Colab session
2. Open `execution/runner.ipynb` in Colab (or upload the project and open the notebook)
3. The notebook will automatically import from these modules

**Note:** The directory structure must be preserved for imports to work.